# 05 - Interpretability

Purpose: inspect model behavior using feature importance and prediction diagnostics.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_PATH = PROJECT_ROOT / 'results' / 'metrics.csv'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Load metrics and trained artifacts

In [2]:
import joblib
from src.visualisation import plot_feature_importance, plot_predicted_vs_actual_scatter

metrics_df = pd.read_csv(METRICS_PATH)
metrics_df

,Model,Task,RMSE,MAE,MAPE,R2,Train_Time_s
0,Random Forest Regressor,RUL,6.373208,5.091485,17.973037,0.962434,0.256819
1,XGBoost Regressor,RUL,6.385948,4.935395,15.623504,0.962284,0.919269
2,Transformer Encoder,RUL,6.583761,5.866489,50.553942,0.951852,5.994190
3,LSTM Neural Network,RUL,33.463212,29.424875,198.333743,-0.243854,1.230259
4,Linear Regression,SOH,0.116027,0.099809,0.127294,0.999769,0.000444
5,XGBoost Regressor,SOH,0.565523,0.367674,0.472972,0.994523,4.504291
6,Random Forest Regressor,SOH,0.714788,0.408414,0.530208,0.991250,0.187762
7,Ridge Regression,SOH,0.943786,0.885303,1.143149,0.984746,0.000350
8,CNN-BiLSTM,SOH,6.894650,6.183861,8.181457,0.013460,0.897369
9,LSTM Neural Network,SOH,6.941440,6.231954,8.022731,0.000024,1.134324


## Feature importance from tree-based RUL model

In [3]:
artifact = joblib.load(MODELS_DIR / 'rul_random_forest_regressor.pkl')
model = artifact['model']
feature_cols = artifact['features']
importance_df = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_})
plot_feature_importance(importance_df, FIG_DIR / '10_best_tree_feature_importance.png', top_n=15)
importance_df.sort_values('importance', ascending=False).head(15)

,feature,importance
14,energy_discharged,0.351793
1,capacity_Ah,0.292316
18,capacity_lag_1,0.131528
19,capacity_lag_3,0.068772
20,capacity_lag_5,0.040515
12,temperature_rise,0.039260
0,cycle_number,0.014902
3,avg_current,0.013822
8,max_voltage,0.009587
9,min_voltage,0.007998


## Predicted vs actual diagnostics (RUL)

In [4]:
feature_df = pd.read_csv(PROCESSED_DIR / 'features_all.csv')
subset = feature_df[feature_df['battery_id'] == 'B0018'].sort_values('cycle_number').copy()
X = subset[feature_cols].fillna(subset[feature_cols].median())
X_scaled = artifact['scaler'].transform(X)
y_pred = model.predict(X_scaled)
y_true = subset['RUL'].to_numpy()
plot_predicted_vs_actual_scatter(y_true, y_pred, 'Predicted vs Actual (RUL - Random Forest)', FIG_DIR / '09_best_model_predicted_vs_actual_scatter.png')
print('Interpretability diagnostics generated.')

Interpretability diagnostics generated.
